In [1]:
# ====================================================
# 1️⃣ 라이브러리 임포트
# ====================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import warnings 
from scipy import stats
from scipy.stats import chi2_contingency, entropy
import time
from functools import wraps
from sklearn.model_selection import StratifiedKFold, KFold, GroupKFold

# FutureWarning 등의 메시지를 깔끔하게 숨기기 위해 추가
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

# skimpy만 사용
try:
    from skimpy import skim
except ImportError:
    print("❌ 라이브러리가 없습니다. 설치해주세요: pip install skimpy")

# missingno 추가
try:
    import missingno as msno
except ImportError:
    print("⚠️ missingno가 없습니다. 설치 권장: pip install missingno")
    msno = None

# 한글 폰트 설정 (Windows/Mac 대응)
import platform
if platform.system() == 'Darwin': # Mac
    plt.rc('font', family='AppleGothic')
else: # Windows
    plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

# ====================================================
# 2️⃣ CONFIGURATION (설정 영역) - 하드코딩 방지
# ====================================================
class CFG:
    # 1. 경로 설정
    BASE_PATH = r"C:\SEMIN\Project\ML_FINAL\1.BASE"  # 데이터 폴더 경로
    OUTPUT_DIR = r"C:\SEMIN\Project\ML_FINAL\0.EDA"  # 결과 저장 경로
    PREPROCESS_DIR = r"C:\SEMIN\Project\ML_FINAL\2.FEATURE"  # 전처리 데이터 저장
    
    # 2. 파일 이름 설정
    TRAIN_FILE = "train_transactions.csv"         # X_train 혹은 통합 train 파일명
    TEST_FILE = "test_transactions.csv"           # X_test 파일명
    TARGET_FILE = "y_train.csv"              # y값이 분리되어 있다면 파일명 입력 (없으면 None)
    
    # 3. 데이터 설정
    TARGET_COL = "gender" 
    GROUP_COL = 'custid'           # 그룹 컬럼명 (GroupKFold 사용 시)
    ID_COL = None                  # ID 컬럼명 (필요시 삭제용)
    
    # 4. 분석 옵션
    CARDINALITY_THRESHOLD = 20       # 시각화할 카테고리 최대 개수
    MIN_CATEGORY_SAMPLES = 30        # 카테고리별 최소 샘플 수 (이하는 제외)
    OUTLIER_METHOD = 'IQR'           # 'IQR' or 'Z-score'
    OUTLIER_THRESHOLD = 1.5          # IQR의 경우 1.5, Z-score의 경우 3
    RANDOM_STATE = 42
    
    # 🆕 추가: EDA 관련
    CLASSIFICATION_THRESHOLD = 20        # 분류/회귀 판단 기준 (타겟 고유값 개수)
    CORRELATION_THRESHOLD = 0.7          # 다중공선성 경고 임계값
    MAX_CATEGORIES_TO_PLOT = 20          # 고카디널리티 시각화 최대 개수
    PLOT_BATCH_SIZE = 4                  # 시각화 배치 크기
    
    # 5. 성능 최적화 설정
    ENABLE_SAMPLING = True           # 대용량 데이터 샘플링 활성화
    SAMPLE_SIZE = 100000             # 샘플링 크기 (행 수)
    SAMPLE_FOR_VIZ_ONLY = True       # True: 시각화만 샘플링 / False: 전체 샘플링
    PARALLEL_PROCESSING = False      # 병렬 처리 (추후 확장 가능)
    
    # 6. 전처리 설정
    AUTO_PREPROCESSING = False        # 자동 전처리 활성화
    MISSING_STRATEGY = {
        'numerical': 'mode',       # 'mean', 'median', 'mode', 'drop'
        'categorical': 'mode'        # 'mode', 'new_category', 'drop'
    }
    SCALING_METHOD = 'robust'        # 'standard', 'minmax', 'robust', None
    ENCODING_METHOD = 'target'       # 'onehot', 'label', 'target', 'frequency'
    OUTLIER_TREATMENT = 'clip'         # 'clip', 'remove', None
    FEATURE_SELECTION = False        # 예측력 낮은 Feature 제거
    FEATURE_SELECTION_THRESHOLD = 0.05  # P-value 임계값
    
    # 🆕 추가: 인코딩 관련
    TARGET_ENCODING_SMOOTHING = 1       # Target Encoding Smoothing Factor
    ONEHOT_MAX_CARDINALITY = 100          # One-Hot Encoding 최대 카디널리티
    CV_FOLDS = 5                         # Target Encoding CV Folds
    USE_GROUP_KFOLD = True               # GroupKFold 사용 여부 (True: 같은 custid는 같은 fold)

    # 🆕 시각화 설정
    ENABLE_VISUALIZATION = False      # 전체 시각화 ON/OFF
    ENABLE_MISSING_VIZ = True        # 결측치 시각화
    ENABLE_OUTLIER_VIZ = True        # 이상치 박스플롯
    ENABLE_DISTRIBUTION_VIZ = True   # 분포 변환 시각화
    ENABLE_CORRELATION_VIZ = True    # 상관관계 히트맵
    ENABLE_TARGET_VIZ = True         # 타겟 분포 시각화
    
    # 🆕 추가: Feature Engineering 옵션
    CREATE_GROUP_FEATURES = True         # 그룹 집계 Feature 생성 여부
    GROUP_AGG_FUNCTIONS = ['mean', 'std', 'count', 'min', 'max']  # 집계 함수
    
    # 🆕 추가: 카테고리 타입 변환 옵션
    AUTO_DETECT_CATEGORICAL = True       # 수치형이지만 카테고리인 변수 자동 감지
    CATEGORICAL_UNIQUE_THRESHOLD = 10    # 고유값이 N개 이하면 카테고리로 간주
    FORCE_CATEGORICAL_COLS = []          # 강제로 카테고리로 취급할 컬럼 리스트
    FORCE_NUMERICAL_COLS = []            # 강제로 수치형으로 취급할 컬럼 리스트
    
    # 🆕 추가: 그룹화 설정 (거래 데이터 → 고객 단위 집계)
    AGGREGATE_BY_GROUP = True           # 그룹으로 자동 집계 활성화 (True: 고객 단위 / False: 거래 단위)
    GROUP_TARGET_AGG = 'mode'           # 타겟 집계 방법 ('first', 'mode', 'mean')
    GROUP_NUM_AGG = ['mean', 'std', 'sum', 'min', 'max', 'count']  # 수치형 집계 함수
    
    # 🆕 추가: 범주형 그룹화 집계 방법
    GROUP_CAT_AGG = 'onehot_sum'               # 'first', 'mode', 'nunique', 'concat', 
                                          # 'onehot_sum', 'onehot_binary', 'onehot_ratio',
                                          # 'frequency', 'entropy'
    ONEHOT_AGG_TOP_K = 10                # One-Hot 집계 시 최대 카테고리 수 (상위 K개만)
    
    # 🆕 추가: 컬럼별 커스텀 집계 (우선순위 최상위)
    CUSTOM_AGG_DICT = {
        # 예시:
        # 'merchant_id': 'onehot_sum',       # 상점별 방문 횟수
        # 'product_category': 'onehot_ratio', # 카테고리별 구매 비율
        # 'payment_method': 'mode',           # 최빈 결제 수단
        # 'transaction_date': ['min', 'max', 'nunique']  # 첫/마지막 거래일, 거래일수
    }

# 폴더 자동 생성
os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
os.makedirs(CFG.PREPROCESS_DIR, exist_ok=True)

# ====================================================
# 3️⃣ 성능 측정 데코레이터
# ====================================================
def timer(func):
    """함수 실행 시간 측정"""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"⏱️ {func.__name__} 실행 시간: {elapsed:.2f}초")
        return result
    return wrapper

# ====================================================
# 4️⃣ 유틸리티 함수 (데이터 로드 & 병합)
# ====================================================
def load_data_custom(filepath):
    """인코딩 에러 방지용 커스텀 로더"""
    if filepath is None or not os.path.exists(filepath):
        return None
        
    try:
        return pd.read_csv(filepath, encoding='utf-8')
    except UnicodeDecodeError:
        try:
            return pd.read_csv(filepath, encoding='cp949')
        except UnicodeDecodeError:
            return pd.read_csv(filepath, encoding='euc-kr')

# ====================================================
# 🆕 그룹화 헬퍼 함수들
# ====================================================
def calculate_entropy_series(series):
    """카테고리 분포의 엔트로피 계산 (다양성 지표)"""
    value_counts = series.value_counts(normalize=True)
    return entropy(value_counts)

def flatten_column_names(df):
    """
    MultiIndex 컬럼을 단일 레벨로 변환
    (sum, mean 등이 붙은 컬럼명 정리)
    🆕 타겟 컬럼은 원래 이름 유지
    """
    new_columns = []
    for col in df.columns:
        if isinstance(col, tuple):
            base_col, agg_func = col
            
            # 🆕 타겟 컬럼은 집계 함수명 제거 (gender_first → gender)
            if base_col == CFG.TARGET_COL:
                new_columns.append(base_col)
            elif isinstance(agg_func, str) and agg_func != '':
                new_columns.append(f"{base_col}_{agg_func}")
            else:
                new_columns.append(base_col)
        else:
            new_columns.append(col)
    
    df.columns = new_columns
    return df

def create_onehot_aggregation(df, col, method='onehot_sum', reference_categories=None):
    """
    범주형 컬럼을 One-Hot 스타일로 집계
    
    Parameters:
    -----------
    df : DataFrame
    col : str - 집계할 범주형 컬럼
    method : str - 'onehot_sum', 'onehot_binary', 'onehot_ratio'
    reference_categories : list - Train에서 결정된 카테고리 리스트 (Test 처리 시 사용)
    
    Returns:
    --------
    DataFrame : custid별 One-Hot 집계 결과
    selected_categories : list - 선택된 카테고리 리스트 (Train에서만 반환)
    """
    
    # 🆕 Train: 상위 K개 카테고리 결정
    if reference_categories is None:
        top_categories = df[col].value_counts().head(CFG.ONEHOT_AGG_TOP_K).index.tolist()
        is_train = True
    else:
        # 🆕 Test: Train의 카테고리 사용
        top_categories = reference_categories
        is_train = False
    
    if method == 'onehot_sum':
        # 각 카테고리 등장 횟수
        result = df[df[col].isin(top_categories)].groupby([CFG.GROUP_COL, col]).size().unstack(fill_value=0)
        # 🆕 Train의 카테고리 순서에 맞춰 컬럼 정렬 (Test에서 누락된 컬럼 추가)
        result = result.reindex(columns=top_categories, fill_value=0)
        result.columns = [f'{col}_count_{cat}' for cat in result.columns]
    
    elif method == 'onehot_binary':
        # 각 카테고리 등장 여부 (0 or 1)
        result = df[df[col].isin(top_categories)].groupby([CFG.GROUP_COL, col]).size().unstack(fill_value=0)
        result = result.reindex(columns=top_categories, fill_value=0)
        result = (result > 0).astype(int)
        result.columns = [f'{col}_has_{cat}' for cat in result.columns]
    
    elif method == 'onehot_ratio':
        # 각 카테고리 비율
        result = df[df[col].isin(top_categories)].groupby([CFG.GROUP_COL, col]).size().unstack(fill_value=0)
        result = result.reindex(columns=top_categories, fill_value=0)
        result = result.div(result.sum(axis=1), axis=0).fillna(0)
        result.columns = [f'{col}_ratio_{cat}' for cat in result.columns]
    
    result = result.reset_index()
    
    if is_train:
        print(f"    ✅ {col}: {method} 생성 ({len(result.columns)-1}개 컬럼)")
        return result, top_categories  # 🆕 Train은 카테고리 리스트도 반환
    else:
        print(f"    ✅ [Test] {col}: {method} 적용 ({len(result.columns)-1}개 컬럼)")
        return result, None

# ====================================================
# 🆕 그룹화 함수 (핵심!)
# ====================================================
@timer
@timer
def aggregate_by_group(df, is_train=True, onehot_categories_map=None):
    """
    거래 데이터 → 고객 단위 집계 (범주형 One-Hot 지원)
    
    Parameters:
    -----------
    onehot_categories_map : dict - Train에서 결정된 One-Hot 카테고리 맵 (Test 처리 시 사용)
    """
    
    if CFG.GROUP_COL not in df.columns:
        print(f"⚠️ 그룹 컬럼 '{CFG.GROUP_COL}'이 데이터에 없습니다. 그룹화를 건너뜁니다.")
        return df, None
    
    print(f"  → 원본: {len(df):,} 행 (거래 단위)")
    
    # ===== 1단계: 기본 집계 딕셔너리 생성 =====
    agg_dict = {}
    onehot_cols = []
    
    # 🔥 추가: Test인데 맵이 없으면 빈 딕셔너리로 초기화
    if not is_train and onehot_categories_map is None:
        onehot_categories_map = {}
        print("⚠️ Test 처리인데 One-Hot 카테고리 맵이 없습니다. One-Hot 집계를 건너뜁니다.")
    
    # 커스텀 설정 우선 적용
    for col, agg_func in CFG.CUSTOM_AGG_DICT.items():
        if col in df.columns and col != CFG.GROUP_COL:
            if agg_func in ['onehot_sum', 'onehot_binary', 'onehot_ratio']:
                onehot_cols.append((col, agg_func))
            else:
                agg_dict[col] = agg_func
    
    # 수치형 변수 집계
    numerical_cols = df.select_dtypes(include=['number']).columns.tolist()
    for col in numerical_cols:
        if col in [CFG.GROUP_COL, CFG.TARGET_COL] or col in agg_dict or col in [c[0] for c in onehot_cols]:
            continue
        agg_dict[col] = CFG.GROUP_NUM_AGG
    
    # 범주형 변수 집계
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    for col in categorical_cols:
        if col == CFG.GROUP_COL or col in agg_dict or col in [c[0] for c in onehot_cols]:
            continue
        
        # 전역 설정에 따라 One-Hot 여부 결정
        if CFG.GROUP_CAT_AGG in ['onehot_sum', 'onehot_binary', 'onehot_ratio']:
            onehot_cols.append((col, CFG.GROUP_CAT_AGG))
        elif CFG.GROUP_CAT_AGG == 'mode':
            agg_dict[col] = lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0]
        elif CFG.GROUP_CAT_AGG == 'first':
            agg_dict[col] = 'first'
        elif CFG.GROUP_CAT_AGG == 'nunique':
            agg_dict[col] = 'nunique'
        elif CFG.GROUP_CAT_AGG == 'concat':
            agg_dict[col] = lambda x: ','.join(x.astype(str).unique())
        elif CFG.GROUP_CAT_AGG == 'frequency':
            agg_dict[col] = lambda x: x.value_counts(normalize=True).iloc[0] if len(x) > 0 else 0
        elif CFG.GROUP_CAT_AGG == 'entropy':
            agg_dict[col] = lambda x: calculate_entropy_series(x)
    
    # 타겟 처리 (Train만)
    if is_train and CFG.TARGET_COL in df.columns:
        if CFG.GROUP_TARGET_AGG == 'first':
            agg_dict[CFG.TARGET_COL] = 'first'
        elif CFG.GROUP_TARGET_AGG == 'mode':
            agg_dict[CFG.TARGET_COL] = lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0]
        elif CFG.GROUP_TARGET_AGG == 'mean':
            agg_dict[CFG.TARGET_COL] = 'mean'
    
    # ===== 2단계: 기본 집계 실행 =====
    print(f"  → 기본 집계 중... ({len(agg_dict)}개 컬럼)")
    df_grouped = df.groupby(CFG.GROUP_COL).agg(agg_dict).reset_index()
    
    # 컬럼명 정리
    df_grouped = flatten_column_names(df_grouped)
    
    # ===== 3단계: One-Hot 스타일 집계 =====
    # 🆕 Train: 카테고리 맵 생성
    if is_train:
        onehot_categories_map = {}
    
    if onehot_cols:
        print(f"  → One-Hot 집계 중... ({len(onehot_cols)}개 컬럼)")
        for col, agg_method in onehot_cols:
            # 🆕 Train: 카테고리 리스트 저장
            if is_train:
                df_onehot, selected_cats = create_onehot_aggregation(df, col, agg_method)
                onehot_categories_map[col] = selected_cats
            else:
                # 🆕 Test: Train 카테고리 사용
                reference_cats = onehot_categories_map.get(col, None)
                df_onehot, _ = create_onehot_aggregation(df, col, agg_method, reference_categories=reference_cats)
            
            df_grouped = pd.merge(df_grouped, df_onehot, on=CFG.GROUP_COL, how='left')
    
    # 🆕 ===== 4단계: 결측치 일괄 처리 =====
    print(f"  → 결측치 처리 시작...")
    
    # 수치형 컬럼의 NaN을 0으로 (타겟 제외)
    numeric_cols = df_grouped.select_dtypes(include=['number']).columns.tolist()
    if is_train and CFG.TARGET_COL in numeric_cols:
        numeric_cols.remove(CFG.TARGET_COL)
    
    # NaN 개수 확인
    nan_counts_before = df_grouped[numeric_cols].isnull().sum().sum()
    
    # 0으로 채우기
    df_grouped[numeric_cols] = df_grouped[numeric_cols].fillna(0)
    
    print(f"  ✅ 결측치 {nan_counts_before:,}개 → 0으로 채움")
    
    # 타겟 컬럼 확인
    if is_train:
        if CFG.TARGET_COL not in df_grouped.columns:
            print(f"⚠️ 경고: 타겟 컬럼 '{CFG.TARGET_COL}'이 사라졌습니다!")
        else:
            print(f"✅ 타겟 컬럼 '{CFG.TARGET_COL}' 정상 유지")
    
    print(f"  → 결과: {len(df_grouped):,} 행 (고객 단위)")
    print(f"  → 컬럼 수: {df.shape[1]} → {df_grouped.shape[1]}")
    
    # 집계 결과 요약
    print("\n📊 집계 결과 요약:")
    print(f"  - 수치형 집계: {len([c for c in df.select_dtypes(include=['number']).columns if c not in [CFG.GROUP_COL, CFG.TARGET_COL]])}개 컬럼")
    print(f"  - 범주형 집계: {len(df.select_dtypes(include=['object', 'category']).columns)}개 컬럼")
    if onehot_cols:
        print(f"  - One-Hot 집계: {len(onehot_cols)}개 컬럼")
    if is_train:
        print(f"  - 타겟 집계: {CFG.GROUP_TARGET_AGG} 방식")
    
    return df_grouped, onehot_categories_map

@timer
def get_full_dataset():
    """
    Train과 Target, Test를 모두 로드하여 반환
    + 그룹화 옵션 지원 (One-Hot 카테고리 맵 전달)
    """
    print(f"📂 데이터 로딩 시작: {CFG.BASE_PATH}")
    
    # ===== Train 로딩 =====
    train_path = os.path.join(CFG.BASE_PATH, CFG.TRAIN_FILE)
    df_train = load_data_custom(train_path)
    
    if df_train is None:
        print("❌ Train 파일을 로드할 수 없습니다.")
        return None, None

    # Target 병합 (Train에만)
    if CFG.TARGET_FILE:
        target_path = os.path.join(CFG.BASE_PATH, CFG.TARGET_FILE)
        df_target = load_data_custom(target_path)
        
        if df_target is not None:
            if CFG.TARGET_COL in df_train.columns:
                df_train = df_train.drop(columns=[CFG.TARGET_COL], errors='ignore')
            
            if CFG.GROUP_COL in df_train.columns and CFG.GROUP_COL in df_target.columns:
                df_train = pd.merge(df_train, df_target, on=CFG.GROUP_COL, how='left')
                print(f"✅ Train-Target 병합 완료 (Key: {CFG.GROUP_COL})")
            else:
                df_train = pd.concat([df_train, df_target], axis=1)
                print("✅ Train-Target 단순 병합 완료")

    if CFG.TARGET_COL not in df_train.columns:
        print(f"❌ 타겟 컬럼 '{CFG.TARGET_COL}'이 Train에 없습니다.")
        return None, None
    
    # 🆕 그룹화 시 카테고리 맵 저장
    onehot_categories_map = None
    
    # 🆕 그룹화 실행 여부 명시적 표시
    if CFG.AGGREGATE_BY_GROUP:
        if CFG.GROUP_COL not in df_train.columns:
            print(f"⚠️ 그룹화 활성화되었으나 '{CFG.GROUP_COL}' 컬럼이 없습니다. 원본 데이터로 진행합니다.")
        else:
            print("\n" + "="*50)
            print(f"👥 {CFG.GROUP_COL}로 데이터 그룹화 시작")
            print("="*50)
            # 🔥 수정: Train 그룹화 시 카테고리 맵 저장
            df_train, onehot_categories_map = aggregate_by_group(df_train, is_train=True)
    else:
        print(f"\n✅ 그룹화 비활성화: 원본 거래 데이터 그대로 사용 ({len(df_train):,}행)")
    
    # ===== Test 로딩 =====
    test_path = os.path.join(CFG.BASE_PATH, CFG.TEST_FILE)
    df_test = load_data_custom(test_path)
    
    if df_test is None:
        print("⚠️ Test 파일을 찾을 수 없습니다. Train만 처리합니다.")
        df_test = None
    else:
        print(f"✅ Test 데이터 로딩 완료: {df_test.shape}")
        
        # Test도 그룹화 (타겟 제외)
        if CFG.AGGREGATE_BY_GROUP and CFG.GROUP_COL in df_test.columns:
            print(f"\n👥 Test 데이터도 {CFG.GROUP_COL}로 그룹화")
            # 🔥 수정: Train의 카테고리 맵 전달
            df_test, _ = aggregate_by_group(df_test, is_train=False, onehot_categories_map=onehot_categories_map)
        elif not CFG.AGGREGATE_BY_GROUP:
            print(f"✅ Test도 원본 거래 데이터 그대로 사용 ({len(df_test):,}행)")
    
    # 메모리 사용량 출력
    memory_train = df_train.memory_usage(deep=True).sum() / 1024**2
    print(f"\n📊 Train shape: {df_train.shape}, 메모리: {memory_train:.2f} MB")
    
    if df_test is not None:
        memory_test = df_test.memory_usage(deep=True).sum() / 1024**2
        print(f"📊 Test shape: {df_test.shape}, 메모리: {memory_test:.2f} MB")
    
    return df_train, df_test

# ====================================================
# 5️⃣ 샘플링 유틸리티
# ====================================================
def smart_sampling(df, sample_size, stratify_col=None):
    """
    대용량 데이터를 효율적으로 샘플링
    - stratify_col: 층화 샘플링 기준 컬럼 (타겟 분포 유지)
    """
    if len(df) <= sample_size:
        return df
    
    if stratify_col and stratify_col in df.columns:
        # 층화 샘플링 (타겟 분포 유지)
        sampled = df.groupby(stratify_col, group_keys=False).apply(
            lambda x: x.sample(min(len(x), int(sample_size * len(x) / len(df))), 
                             random_state=CFG.RANDOM_STATE)
        )
    else:
        # 단순 랜덤 샘플링
        sampled = df.sample(n=sample_size, random_state=CFG.RANDOM_STATE)
    
    print(f"🔽 샘플링 적용: {len(df):,} → {len(sampled):,} 행")
    return sampled

# ====================================================
# 6️⃣ EDA 파이프라인 클래스
# ====================================================
class EDAPipeline:
    def __init__(self, df, target_col):
        self.df_full = df.copy()
        self.df = df.copy()
        self.target_col = target_col
    
        # 🆕 ID/GROUP 컬럼 제외 (분석 전 제거)
        cols_to_exclude = []
        if CFG.ID_COL and CFG.ID_COL in self.df.columns:
            cols_to_exclude.append(CFG.ID_COL)
        if CFG.GROUP_COL and CFG.GROUP_COL in self.df.columns:
         cols_to_exclude.append(CFG.GROUP_COL)
    
        if cols_to_exclude:
            print(f"ℹ️ ID/GROUP 컬럼 제외: {cols_to_exclude}")
            self.df = self.df.drop(columns=cols_to_exclude)
            self.df_full = self.df_full.drop(columns=cols_to_exclude)
    
        # 샘플링 적용 (시각화용)
        if CFG.ENABLE_SAMPLING and len(self.df) > CFG.SAMPLE_SIZE:
            if CFG.SAMPLE_FOR_VIZ_ONLY:
                print("🔽 시각화용 샘플링 데이터 생성 중...")
                self.df_viz = smart_sampling(self.df, CFG.SAMPLE_SIZE, stratify_col=target_col)
            else:
                print("🔽 전체 분석용 샘플링 적용 중...")
                self.df = smart_sampling(self.df, CFG.SAMPLE_SIZE, stratify_col=target_col)
                self.df_viz = self.df
        else:
            self.df_viz = self.df
    
        # 🆕 카테고리 타입 자동 감지
        self._detect_categorical_types()
    
        # 🔥 수정: ID/GROUP이 제외된 후 categorical_cols 생성
        self.categorical_cols = self.df.select_dtypes(include=['object', 'category']).columns.tolist()
        initial_numerical_cols = self.df.select_dtypes(include=['number']).columns.tolist()
    
        self.numerical_cols = []
    
        # 수치형 컬럼 중 이진 컬럼 (Cardinality=2) 제외 및 범주형으로 이동
        for col in initial_numerical_cols:
            if col == self.target_col:  # 🔥 ID_COL 체크 제거 (이미 제외됨)
                self.numerical_cols.append(col)
                continue
            
            # 🆕 강제 설정 확인
            if col in CFG.FORCE_CATEGORICAL_COLS:
                self.categorical_cols.append(col)
                print(f"✅ '{col}'을(를) 강제로 범주형으로 설정했습니다.")
                continue
            
            if col in CFG.FORCE_NUMERICAL_COLS:
                self.numerical_cols.append(col)
                print(f"✅ '{col}'을(를) 강제로 수치형으로 설정했습니다.")
                continue
            
            # 자동 감지: 고유값 2개 또는 설정값 이하면 범주형으로 이동
            n_unique = df[col].nunique()
            if n_unique == 2:
                self.categorical_cols.append(col) 
                print(f"✅ Cardinality=2인 '{col}'을(를) 수치형 분석에서 제외하고 범주형으로 이동했습니다.")
            elif CFG.AUTO_DETECT_CATEGORICAL and n_unique <= CFG.CATEGORICAL_UNIQUE_THRESHOLD:
                self.categorical_cols.append(col)
                print(f"✅ 고유값 {n_unique}개인 '{col}'을(를) 범주형으로 자동 감지했습니다.")
            else:
                self.numerical_cols.append(col)
        
        # 최종 ID 컬럼 및 타겟 컬럼을 수치형 리스트에서 제외
        if CFG.ID_COL in self.numerical_cols:
             self.numerical_cols.remove(CFG.ID_COL)
        if target_col in self.numerical_cols:
            self.numerical_cols.remove(target_col)
            
        self.is_classification = self.df[self.target_col].nunique() < CFG.CLASSIFICATION_THRESHOLD
        
        # 분석 결과 저장용
        self.missing_info = {}
        self.outlier_info = {}
        self.transform_recommendations = {}
        self.feature_importance = {}
    
    def _detect_categorical_types(self):
        """🆕 수치형이지만 실제로는 카테고리인 변수 감지"""
        if not CFG.AUTO_DETECT_CATEGORICAL:
            return
        
        print("\n" + "="*50)
        print("🔍 카테고리 타입 자동 감지 시작")
        print("="*50)
        
        # object/category 타입이 아닌 수치형 컬럼 중에서 검사
        numerical_cols = self.df.select_dtypes(include=['number']).columns.tolist()
        
        for col in numerical_cols:
            if col in [self.target_col, CFG.ID_COL, CFG.GROUP_COL]:
                continue
            
            n_unique = self.df[col].nunique()
            
            # 고유값이 임계값 이하면 카테고리로 변환
            if n_unique <= CFG.CATEGORICAL_UNIQUE_THRESHOLD:
                # 실제 데이터 타입 변환
                self.df[col] = self.df[col].astype('category')
                self.df_full[col] = self.df_full[col].astype('category')
                if hasattr(self, 'df_viz'):
                    self.df_viz[col] = self.df_viz[col].astype('category')
                
                print(f"  → '{col}': 고유값 {n_unique}개 → category 타입으로 변환")
        
    @timer
    def show_basic_info(self):
        """기본 정보 및 Skimpy 요약"""
        print("\n" + "="*50)
        print("1. 기본 컬럼 정보 (Dtypes)")
        print("="*50)
        print(self.df.info())
        
        print("\n" + "="*50)
        print("2. Skimpy Summary (깔끔한 요약)")
        print("="*50)
        skim(self.df)

    @timer
    def analyze_missing_values(self):
        """결측치 분석 및 시각화"""
        print("\n" + "="*50)
        print("3. 결측치(Missing Values) 분석")
        print("="*50)
        
        missing_df = pd.DataFrame({
            'Column': self.df.columns,
            'Missing_Count': self.df.isnull().sum(),
            'Missing_Percent': (self.df.isnull().sum() / len(self.df) * 100).round(2)
        }).sort_values('Missing_Count', ascending=False)
        
        missing_df = missing_df[missing_df['Missing_Count'] > 0]
        
        if len(missing_df) == 0:
            print("✅ 결측치가 없습니다!")
            return
        
        print(missing_df.to_string(index=False))
        
        # 🆕 시각화 제어 추가
        if not CFG.ENABLE_VISUALIZATION or not CFG.ENABLE_MISSING_VIZ:
            print("⚠️ 결측치 시각화 비활성화됨")
            self.missing_info = missing_df
            return
        
        # missingno 시각화 (샘플링 데이터 사용)
        if msno is not None:
            print("\n📊 결측치 패턴 시각화 중...")
            
            # Matrix plot
            plt.figure(figsize=(12, 6))
            msno.matrix(self.df_viz, sparkline=False, fontsize=10)
            plt.title("결측치 패턴 Matrix", fontsize=14, pad=20)
            save_path = f"{CFG.OUTPUT_DIR}/missing_matrix.png"
            plt.savefig(save_path, bbox_inches='tight')
            plt.show()
            print(f"saved: {save_path}")
            
            # Heatmap (상관관계)
            if len(missing_df) > 1:
                plt.figure(figsize=(10, 8))
                msno.heatmap(self.df_viz, fontsize=10)
                plt.title("결측치 간 상관관계 Heatmap", fontsize=14, pad=20)
                save_path = f"{CFG.OUTPUT_DIR}/missing_heatmap.png"
                plt.savefig(save_path, bbox_inches='tight')
                plt.show()
                print(f"saved: {save_path}")
        
        self.missing_info = missing_df

    @timer
    def detect_outliers(self):
        """이상치 탐지 (IQR or Z-score)"""
        print("\n" + "="*50)
        print(f"4. 이상치(Outliers) 탐지 (Method: {CFG.OUTLIER_METHOD})")
        print("="*50)
        
        if not self.numerical_cols:
            print("🚫 수치형 변수가 없어 이상치 탐지를 건너뜁니다.")
            return
        
        outlier_summary = []
        
        for col in self.numerical_cols:
            data = self.df[col].dropna()
            
            if CFG.OUTLIER_METHOD == 'IQR':
                Q1 = data.quantile(0.25)
                Q3 = data.quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - CFG.OUTLIER_THRESHOLD * IQR
                upper_bound = Q3 + CFG.OUTLIER_THRESHOLD * IQR
                outliers = data[(data < lower_bound) | (data > upper_bound)]
                
            elif CFG.OUTLIER_METHOD == 'Z-score':
                z_scores = np.abs(stats.zscore(data))
                outliers = data[z_scores > CFG.OUTLIER_THRESHOLD]
            
            outlier_count = len(outliers)
            outlier_pct = (outlier_count / len(data) * 100)
            
            outlier_summary.append({
                'Feature': col,
                'Outlier_Count': outlier_count,
                'Outlier_Percent': round(outlier_pct, 2),
                'Min': data.min(),
                'Max': data.max(),
                'Lower_Bound': lower_bound if CFG.OUTLIER_METHOD == 'IQR' else None,
                'Upper_Bound': upper_bound if CFG.OUTLIER_METHOD == 'IQR' else None
            })
        
        df_outliers = pd.DataFrame(outlier_summary).sort_values('Outlier_Count', ascending=False)
        print(df_outliers.to_string(index=False))
        
        self.outlier_info = df_outliers
        
        # 🆕 시각화 제어 추가
        if not CFG.ENABLE_VISUALIZATION or not CFG.ENABLE_OUTLIER_VIZ:
            print("⚠️ 이상치 시각화 비활성화됨")
            return
        
        # Boxplot으로 이상치 시각화 (샘플링 데이터 사용)
        print("\n📊 이상치 Boxplot 시각화 중...")
        num_cols_to_plot = [col for col in self.numerical_cols]
        
        for i in range(0, len(num_cols_to_plot), CFG.PLOT_BATCH_SIZE):
            cols_batch = num_cols_to_plot[i:i+CFG.PLOT_BATCH_SIZE]
            fig, axes = plt.subplots(1, len(cols_batch), figsize=(20, 5))
            if len(cols_batch) == 1:
                axes = [axes]
            
            for ax, col in zip(axes, cols_batch):
                sns.boxplot(y=self.df_viz[col].dropna(), ax=ax, color='skyblue')
                ax.set_title(f'{col} Boxplot', fontsize=12)
                ax.set_ylabel('')
            
            plt.tight_layout()
            save_path = f"{CFG.OUTPUT_DIR}/outliers_boxplot_group_{i//CFG.PLOT_BATCH_SIZE + 1}.png"
            plt.savefig(save_path)
            plt.show()
            print(f"saved: {save_path}")

    @timer
    def analyze_numerical_distributions(self):
        """수치형 변수 분포 분석 (변환 전후 비교 시각화)"""
        print("\n" + "="*50)
        print("5. 수치형 변수 분포 분석 및 변환 효과 비교")
        print("="*50)

        if not self.numerical_cols:
            print("🚫 분석할 수치형 변수가 없습니다.")
            return

        transform_recommendations = []
        
        for col in self.numerical_cols:
            series = self.df[col].dropna()
            series_viz = self.df_viz[col].dropna()  # 시각화용
            
            # 음수 확인
            has_negative = (series < 0).any()
            min_value = series.min()
            
            # 원본 통계
            raw_skew = series.skew()
            raw_kurt = series.kurtosis()
            
            # ===== Log 변환 (음수 처리 개선) =====
            if has_negative:
                shifted_series = series - min_value + 1
                log_series = np.log1p(shifted_series)
                log_skew = log_series.skew()
                log_method = 'shifted'
                if CFG.ENABLE_VISUALIZATION and CFG.ENABLE_DISTRIBUTION_VIZ:
                    print(f"  → {col}: 음수 존재 ({min_value:.2f}), Shift 후 Log 변환 시도")
            else:
                log_series = np.log1p(series)
                log_skew = log_series.skew()
                log_method = 'direct'
            
            # ===== Sqrt 변환 =====
            sqrt_series = np.sqrt(series - min_value + 1)
            sqrt_skew = sqrt_series.skew()
            
            # 최적 변환 결정
            skew_dict = {'Raw': abs(raw_skew), 'Log': abs(log_skew), 'Sqrt': abs(sqrt_skew)}
            best_transform = min(skew_dict, key=skew_dict.get)
            
            transform_recommendations.append({
                'Feature': col,
                'Raw_Skew': round(raw_skew, 4),
                'Log_Skew': round(log_skew, 4),
                'Sqrt_Skew': round(sqrt_skew, 4),
                'Best_Transform': best_transform,
                'Has_Negative': has_negative,
                'Min_Value': round(min_value, 4),
                'Log_Method': log_method if best_transform == 'Log' else None
            })
            
            # 🆕 시각화 제어 추가
            if not CFG.ENABLE_VISUALIZATION or not CFG.ENABLE_DISTRIBUTION_VIZ:
                continue
            
            # ===== 변환 전후 분포 시각화 =====
            fig, axes = plt.subplots(1, 4, figsize=(20, 4))
            
            # 1. 원본 분포
            sns.histplot(series_viz, kde=True, ax=axes[0], color='skyblue')
            axes[0].axvline(series_viz.median(), color='r', linestyle='--', 
                           label=f'Median: {series_viz.median():.2f}')
            axes[0].set_title(f'{col} - Raw (Skew: {raw_skew:.3f})', fontsize=11)
            axes[0].legend()
            
            # 2. Log 변환
            if has_negative:
                shifted_series_viz = series_viz - min_value + 1
                log_series_viz = np.log1p(shifted_series_viz)
                sns.histplot(log_series_viz, kde=True, ax=axes[1], color='lightgreen')
                axes[1].set_title(f'{col} - Log (Shifted, Skew: {log_skew:.3f})', fontsize=11)
            else:
                log_series_viz = np.log1p(series_viz)
                sns.histplot(log_series_viz, kde=True, ax=axes[1], color='lightgreen')
                axes[1].set_title(f'{col} - Log Transform (Skew: {log_skew:.3f})', fontsize=11)
            
            # 3. Sqrt 변환
            sqrt_series_viz = np.sqrt(series_viz - min_value + 1)
            sns.histplot(sqrt_series_viz, kde=True, ax=axes[2], color='lightcoral')
            axes[2].set_title(f'{col} - Sqrt Transform (Skew: {sqrt_skew:.3f})', fontsize=11)
            
            # 4. QQ Plot
            stats.probplot(series_viz, dist="norm", plot=axes[3])
            axes[3].set_title(f'{col} - Q-Q Plot', fontsize=11)
            
            plt.tight_layout()
            save_path = f"{CFG.OUTPUT_DIR}/dist_transform_{col}.png"
            plt.savefig(save_path)
            plt.show()
            print(f"saved: {save_path}")
        
        # 변환 추천 요약표
        df_transform = pd.DataFrame(transform_recommendations)
        print("\n✨ 변환 기법 추천 요약")
        print(df_transform.to_string(index=False))
        
        self.transform_recommendations = df_transform
        return df_transform

    @timer
    def analyze_correlation(self):
        """상관관계 분석 (다중공선성 체크)"""
        print("\n" + "="*50)
        print("6. 상관관계 분석 (Correlation Heatmap)")
        print("="*50)
        
        if len(self.numerical_cols) < 2:
            print("🚫 수치형 변수가 2개 미만이어서 상관관계 분석을 건너뜁니다.")
            return
        
        # 타겟 포함 상관관계 (샘플링 데이터 사용)
        corr_cols = self.numerical_cols + [self.target_col]
        corr_matrix = self.df_viz[corr_cols].corr()
        
        # 🆕 시각화 제어 추가
        if CFG.ENABLE_VISUALIZATION and CFG.ENABLE_CORRELATION_VIZ:
            # Heatmap 시각화
            plt.figure(figsize=(12, 10))
            mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
            sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
                        cmap='coolwarm', center=0, square=True, linewidths=1,
                        cbar_kws={"shrink": 0.8})
            plt.title('Feature 간 상관관계 (Correlation Heatmap)', fontsize=14, pad=20)
            plt.tight_layout()
            save_path = f"{CFG.OUTPUT_DIR}/correlation_heatmap.png"
            plt.savefig(save_path)
            plt.show()
            print(f"saved: {save_path}")
        else:
            print("⚠️ 상관관계 시각화 비활성화됨")
        
        # 높은 상관관계 쌍 추출 (다중공선성 경고)
        print(f"\n⚠️ 높은 상관관계 (|corr| > {CFG.CORRELATION_THRESHOLD}) - 다중공선성 주의")
        high_corr_pairs = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                if abs(corr_matrix.iloc[i, j]) > CFG.CORRELATION_THRESHOLD:
                    high_corr_pairs.append({
                        'Feature_1': corr_matrix.columns[i],
                        'Feature_2': corr_matrix.columns[j],
                        'Correlation': round(corr_matrix.iloc[i, j], 4)
                    })
        
        if high_corr_pairs:
            df_high_corr = pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False, key=abs)
            print(df_high_corr.to_string(index=False))
        else:
            print("✅ 높은 상관관계를 가진 Feature 쌍이 없습니다.")
    
    @timer
    def analyze_cardinality(self):
        """카디널리티(고유값 개수) 확인"""
        print("\n" + "="*50)
        print("7. 카디널리티 분석 (범주형 변수)")
        print("="*50)
        
        if not self.categorical_cols:
            print("🚫 데이터셋에 dtype이 'object' 또는 'category'인 범주형 변수가 없습니다.")
            return pd.DataFrame() 

        cardinality_data = []
        for col in self.categorical_cols:
            n_unique = self.df[col].nunique()
            unique_vals = self.df[col].unique()[:5]
            cardinality_data.append({'Feature': col, 'Cardinality': n_unique, 'Examples': unique_vals})
            
        df_card = pd.DataFrame(cardinality_data).sort_values('Cardinality', ascending=False)
        print(df_card.to_string(index=False))
        return df_card

    @timer
    def analyze_feature_importance(self):
        """Feature 예측력 분석 (통계적 검정)"""
        print("\n" + "="*50)
        print("8. Feature 예측력 분석 (Statistical Tests)")
        print("="*50)
        
        importance_results = []
        
        # A. 수치형 변수 - ANOVA F-test (분류) or Correlation (회귀)
        if self.numerical_cols:
            print("\n[수치형 변수 예측력]")
            for col in self.numerical_cols:
                if self.is_classification:
                    # ANOVA F-test
                    groups = [self.df[self.df[self.target_col] == cat][col].dropna() 
                             for cat in self.df[self.target_col].unique()]
                    f_stat, p_value = stats.f_oneway(*groups)
                    importance_results.append({
                        'Feature': col,
                        'Type': 'Numerical',
                        'Test': 'ANOVA F-test',
                        'Statistic': round(f_stat, 4),
                        'P-value': p_value,
                        'Significant': '⭐' if p_value < 0.05 else ''
                    })
                else:
                    # Pearson Correlation
                    corr, p_value = stats.pearsonr(self.df[col].dropna(), 
                                                   self.df[self.target_col].dropna())
                    importance_results.append({
                        'Feature': col,
                        'Type': 'Numerical',
                        'Test': 'Pearson Corr',
                        'Statistic': round(corr, 4),
                        'P-value': p_value,
                        'Significant': '⭐' if p_value < 0.05 else ''
                    })
        
        # B. 범주형 변수 - Chi-square test (분류만)
        if self.categorical_cols and self.is_classification:
            print("\n[범주형 변수 예측력]")
            for col in self.categorical_cols:
                contingency_table = pd.crosstab(self.df[col], self.df[self.target_col])
                chi2, p_value, dof, expected = chi2_contingency(contingency_table)
                
                # Cramer's V (효과 크기)
                n = contingency_table.sum().sum()
                cramers_v = np.sqrt(chi2 / (n * (min(contingency_table.shape) - 1)))
                
                importance_results.append({
                    'Feature': col,
                    'Type': 'Categorical',
                    'Test': 'Chi-square',
                    'Statistic': round(chi2, 4),
                    'P-value': p_value,
                    'Cramers_V': round(cramers_v, 4),
                    'Significant': '⭐' if p_value < 0.05 else ''
                })
        
        df_importance = pd.DataFrame(importance_results)
        df_importance['P-value'] = df_importance['P-value'].apply(lambda x: f'{x:.4e}')
        print("\n📊 Feature 예측력 순위 (P-value 기준)")
        print(df_importance.to_string(index=False))
        
        self.feature_importance = df_importance
        return df_importance


    

    @timer
    def plot_target_distributions(self):
        """
        범주형/수치형 변수의 타겟별 분포 시각화
        """
        print("\n" + "="*50)
        print("9. Feature 유형별 타겟 분포 시각화")
        print("="*50)

        # 🆕 시각화 제어 추가
        if not CFG.ENABLE_VISUALIZATION or not CFG.ENABLE_TARGET_VIZ:
            print("⚠️ 타겟 분포 시각화 비활성화됨")
            return

        # 샘플링 데이터 사용
        df = self.df_viz
        
        # 범주형 변수 시각화
        if self.categorical_cols:
            print("\n[범주형 변수 시각화]")
            for col in self.categorical_cols:
                n_unique = df[col].nunique()
                
                # 고카디널리티: 샘플 수 필터링 + Violinplot
                if n_unique > CFG.CARDINALITY_THRESHOLD:
                    print(f"🔄 {col} (고유값 {n_unique}개) - 샘플 수 필터링 후 Violinplot")
                    
                    category_counts = df[col].value_counts()
                    valid_categories = category_counts[category_counts >= CFG.MIN_CATEGORY_SAMPLES].index
                    df_filtered = df[df[col].isin(valid_categories)]
                    
                    if len(valid_categories) == 0:
                        print(f"⚠️ {col}: 최소 샘플 수({CFG.MIN_CATEGORY_SAMPLES})를 만족하는 카테고리가 없습니다.")
                        continue
                    
                    plt.figure(figsize=(14, 6))
                    
                    if len(valid_categories) > CFG.MAX_CATEGORIES_TO_PLOT:
                        top_categories = category_counts[category_counts >= CFG.MIN_CATEGORY_SAMPLES].head(CFG.MAX_CATEGORIES_TO_PLOT).index
                        df_filtered = df_filtered[df_filtered[col].isin(top_categories)]
                        print(f"  → 상위 {CFG.MAX_CATEGORIES_TO_PLOT}개 카테고리만 시각화")
                    
                    sns.violinplot(data=df_filtered, x=col, y=self.target_col, 
                                 palette='muted', inner='box')
                    plt.xticks(rotation=45, ha='right')
                    plt.title(f"{col}별 타겟 분포 (Violinplot)", fontsize=14)
                    plt.ylabel(self.target_col)
                    plt.tight_layout()
                    
                    save_path = f"{CFG.OUTPUT_DIR}/dist_cat_HC_{col}_violin.png"
                    plt.savefig(save_path)
                    plt.show()
                    print(f"saved: {save_path}")
                    continue
                
                # 저카디널리티: Countplot
                plt.figure(figsize=(10, 5))
                sns.countplot(data=df, x=col, hue=self.target_col, palette='Set2')
                plt.title(f"{col}별 타겟 분포 (Cardinality: {n_unique})", fontsize=14)
                plt.xticks(rotation=45, ha='right')
                plt.tight_layout()
                
                save_path = f"{CFG.OUTPUT_DIR}/dist_cat_{col}_vs_{self.target_col}.png"
                plt.savefig(save_path)
                plt.show()
                print(f"saved: {save_path}")

        # 수치형 변수 시각화
        if self.numerical_cols:
            print("\n[수치형 변수 시각화]")
            for i, col in enumerate(self.numerical_cols):
                
                if i % 2 == 0:
                    plt.figure(figsize=(20, 5))

                # KDE Plot
                plt.subplot(1, 4, (i % 2) * 2 + 1)
                if self.is_classification:
                    sns.kdeplot(data=df, x=col, hue=self.target_col, 
                              fill=True, common_norm=False, palette='Set1', alpha=0.5)
                    plt.title(f'{col} - KDE (타겟별 분포)', fontsize=12)
                else:
                    sns.scatterplot(data=df, x=col, y=self.target_col, alpha=0.5)
                    plt.title(f'{col} - Scatter', fontsize=12)
                
                # Boxplot
                plt.subplot(1, 4, (i % 2) * 2 + 2)
                if self.is_classification:
                    sns.boxplot(data=df, x=self.target_col, y=col, palette='Set2')
                    plt.title(f'{col} - Boxplot (타겟별)', fontsize=12)
                else:
                    sns.boxplot(y=df[col], color='skyblue')
                    plt.title(f'{col} - Boxplot', fontsize=12)
                
                if (i + 1) % 2 == 0 or (i + 1) == len(self.numerical_cols):
                    plt.tight_layout()
                    save_path = f"{CFG.OUTPUT_DIR}/dist_num_target_group_{i//2 + 1}.png"
                    plt.savefig(save_path)
                    plt.show()
                    print(f"saved: {save_path}")


# ====================================================
# 7️⃣ 자동 전처리 파이프라인 (Train/Test 통합)
# ====================================================
class AutoPreprocessor:
    """
    Train과 Test를 함께 처리하는 전처리 파이프라인
    Train에서 학습한 통계량을 Test에 적용 (Data Leakage 방지)
    """
    def __init__(self, df_train, eda_pipeline, df_test=None):
        self.df_train = df_train.copy()
        self.df_test = df_test.copy() if df_test is not None else None
        self.eda = eda_pipeline
        self.target_col = eda_pipeline.target_col
        
        # 🆕 ID 컬럼 백업 (조건부: 그룹화 안 했을 때만)
        self.train_id = None
        self.test_id = None
        
        # 🔥 그룹화 안 했을 때만 ID 백업
        if not CFG.AGGREGATE_BY_GROUP:
            if CFG.ID_COL and CFG.ID_COL in self.df_train.columns:
                self.train_id = self.df_train[CFG.ID_COL].copy()
                print(f"💾 Train ID 컬럼 '{CFG.ID_COL}' 백업 완료 (그룹화 안 함)")
            
            if self.df_test is not None and CFG.ID_COL and CFG.ID_COL in self.df_test.columns:
                self.test_id = self.df_test[CFG.ID_COL].copy()
                print(f"💾 Test ID 컬럼 '{CFG.ID_COL}' 백업 완료 (그룹화 안 함)")
        else:
            print(f"🚫 그룹화 활성화: ID 컬럼 '{CFG.ID_COL}' 백업 건너뜀 (집계 후 의미 없음)")
        
        # 통계량 저장용 딕셔너리
        self.stats = {
            'missing_fill_values': {},
            'outlier_bounds': {},
            'target_encoding_maps': {},
            'scaler': None,
            'group_agg_stats': {},
            'transform_min_values': {}
        }
        
    @timer
    def handle_missing_values(self):
        print("\n" + "="*50)
        print("🔧 결측치 처리 시작")
        print("="*50)
        
        # ===== Train 처리 & 통계량 저장 =====
        for col in self.eda.numerical_cols:
            if self.df_train[col].isnull().sum() > 0:
                if CFG.MISSING_STRATEGY['numerical'] == 'mean':
                    fill_value = self.df_train[col].mean()
                elif CFG.MISSING_STRATEGY['numerical'] == 'median':
                    fill_value = self.df_train[col].median()
                elif CFG.MISSING_STRATEGY['numerical'] == 'mode':
                    fill_value = self.df_train[col].mode()[0]
                else:
                    continue
                
                self.stats['missing_fill_values'][col] = fill_value
                self.df_train[col].fillna(fill_value, inplace=True)
                print(f"✅ [Train] {col}: {CFG.MISSING_STRATEGY['numerical']} = {fill_value:.4f}")
        
        for col in self.eda.categorical_cols:
            if self.df_train[col].isnull().sum() > 0:
                if CFG.MISSING_STRATEGY['categorical'] == 'mode':
                    fill_value = self.df_train[col].mode()[0]
                    self.stats['missing_fill_values'][col] = fill_value
                    self.df_train[col].fillna(fill_value, inplace=True)
                elif CFG.MISSING_STRATEGY['categorical'] == 'new_category':
                    self.stats['missing_fill_values'][col] = 'MISSING'
                    self.df_train[col].fillna('MISSING', inplace=True)
                print(f"✅ [Train] {col}: {fill_value}")
        
        # ===== Test 처리 (Train 통계량 사용) =====
        if self.df_test is not None:
            print("\n📦 Test 데이터 결측치 처리 (Train 통계량 적용)")
            for col, fill_value in self.stats['missing_fill_values'].items():
                if col in self.df_test.columns and self.df_test[col].isnull().sum() > 0:
                    self.df_test[col].fillna(fill_value, inplace=True)
                    print(f"✅ [Test] {col}: {fill_value}")
        
        return self.df_train, self.df_test
    
    @timer
    def handle_outliers(self):
        """이상치 처리 (Train 기준 → Test 적용)"""
        print("\n" + "="*50)
        print("🔧 이상치 처리 시작")
        print("="*50)
        
        if CFG.OUTLIER_TREATMENT is None:
            print("⚠️ 이상치 처리 비활성화")
            return self.df_train, self.df_test
        
        # ===== Train에서 경계값 계산 =====
        for _, row in self.eda.outlier_info.iterrows():
            col = row['Feature']
            
            Q1 = self.df_train[col].quantile(0.25)
            Q3 = self.df_train[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - CFG.OUTLIER_THRESHOLD * IQR
            upper_bound = Q3 + CFG.OUTLIER_THRESHOLD * IQR
            
            self.stats['outlier_bounds'][col] = (lower_bound, upper_bound)
            
            if CFG.OUTLIER_TREATMENT == 'clip':
                self.df_train[col] = self.df_train[col].clip(lower_bound, upper_bound)
                print(f"✅ [Train] {col}: Clipping ({lower_bound:.2f} ~ {upper_bound:.2f})")
            
            elif CFG.OUTLIER_TREATMENT == 'remove':
                before_len = len(self.df_train)
                self.df_train = self.df_train[(self.df_train[col] >= lower_bound) & 
                                              (self.df_train[col] <= upper_bound)]
                print(f"✅ [Train] {col}: {before_len - len(self.df_train)}개 행 제거")
        
        # ===== Test 처리 (Train 경계값 사용) =====
        if self.df_test is not None and CFG.OUTLIER_TREATMENT == 'clip':
            print("\n📦 Test 데이터 이상치 처리 (Train 경계값 적용)")
            for col, (lower, upper) in self.stats['outlier_bounds'].items():
                if col in self.df_test.columns:
                    self.df_test[col] = self.df_test[col].clip(lower, upper)
                    print(f"✅ [Test] {col}: Clipping ({lower:.2f} ~ {upper:.2f})")
        
        return self.df_train, self.df_test
    
    @timer
    def create_group_features(self):
        """그룹 집계 Feature 생성 (Train → Test)"""
        if not CFG.CREATE_GROUP_FEATURES or not CFG.GROUP_COL:
            print("⚠️ 그룹 Feature 생성 비활성화")
            return self.df_train, self.df_test
        
        # 🆕 이미 그룹화된 데이터면 경고
        if CFG.AGGREGATE_BY_GROUP:
            print("⚠️ 데이터가 이미 그룹화되어 있어 그룹 Feature 생성을 건너뜁니다.")
            return self.df_train, self.df_test
        
        print("\n" + "="*50)
        print("🔧 그룹 집계 Feature 생성 시작 (거래 데이터 기준)")
        print("="*50)
        
        # ===== Train에서 그룹 통계량 계산 =====
        for col in self.eda.numerical_cols:
            for agg_func in CFG.GROUP_AGG_FUNCTIONS:
                agg_col_name = f'{CFG.GROUP_COL}_{agg_func}_{col}'
                
                # Train 그룹별 통계량 계산
                group_stats = self.df_train.groupby(CFG.GROUP_COL)[col].agg(agg_func)
                self.stats['group_agg_stats'][agg_col_name] = group_stats
                
                # Train에 적용
                self.df_train[agg_col_name] = self.df_train[CFG.GROUP_COL].map(group_stats)
                print(f"✅ [Train] {agg_col_name} 생성")
        
        # 거래 건수
        transaction_counts = self.df_train.groupby(CFG.GROUP_COL).size()
        self.stats['group_agg_stats']['transaction_count'] = transaction_counts
        self.df_train[f'{CFG.GROUP_COL}_transaction_count'] = self.df_train[CFG.GROUP_COL].map(transaction_counts)
        
        # ===== Test에 적용 (Train 통계량 사용) =====
        if self.df_test is not None:
            print("\n📦 Test 데이터 그룹 Feature 생성 (Train 통계량 적용)")
            for agg_col_name, group_stats in self.stats['group_agg_stats'].items():
                if agg_col_name != 'transaction_count':
                    # 새로운 custid는 전체 평균으로 대체
                    global_mean = group_stats.mean()
                    self.df_test[agg_col_name] = self.df_test[CFG.GROUP_COL].map(group_stats).fillna(global_mean)
                else:
                    # 거래 건수
                    self.df_test[f'{CFG.GROUP_COL}_transaction_count'] = self.df_test[CFG.GROUP_COL].map(group_stats).fillna(0)
                
                print(f"✅ [Test] {agg_col_name} 적용")
        
        return self.df_train, self.df_test
    
    
    @timer
    def apply_transformations(self):
        """변수 변환 (Train/Test 동일 적용, 음수 처리 개선)"""
        print("\n" + "="*50)
        print("🔧 변수 변환 적용 시작")
        print("="*50)
        
        for _, row in self.eda.transform_recommendations.iterrows():
            col = row['Feature']
            best_transform = row['Best_Transform']
            has_negative = row['Has_Negative']
            min_value = row['Min_Value']
            
            # ===== Log 변환 (음수 처리 개선) =====
            if best_transform == 'Log':
                # 음수가 있으면 Shift 후 Log
                if has_negative:
                    self.stats['transform_min_values'][col] = min_value
                    
                    # Train
                    shifted_train = self.df_train[col] - min_value + 1
                    shifted_train = shifted_train.clip(lower=0.001)  # 🔥 음수 방지
                    self.df_train[f'{col}_log'] = np.log1p(shifted_train)
                    
                    # Test
                    if self.df_test is not None:
                        shifted_test = self.df_test[col] - min_value + 1
                        shifted_test = shifted_test.clip(lower=0.001)  # 🔥 음수 방지
                        self.df_test[f'{col}_log'] = np.log1p(shifted_test)
                    
                    print(f"✅ {col}: Log 변환 적용 (Shift: {-min_value+1:.2f}, Train & Test)")
                
                # 음수 없으면 그냥 Log
                else:
                    self.df_train[f'{col}_log'] = np.log1p(self.df_train[col])
                    if self.df_test is not None:
                        self.df_test[f'{col}_log'] = np.log1p(self.df_test[col])
                    print(f"✅ {col}: Log 변환 적용 (Direct, Train & Test)")
            
            # ===== Sqrt 변환 =====
            elif best_transform == 'Sqrt':
                self.stats['transform_min_values'][col] = min_value
                
                # Train
                shifted_train = self.df_train[col] - min_value + 1
                shifted_train = shifted_train.clip(lower=0)  # 🔥 음수 방지
                self.df_train[f'{col}_sqrt'] = np.sqrt(shifted_train)
                
                # Test
                if self.df_test is not None:
                    shifted_test = self.df_test[col] - min_value + 1
                    shifted_test = shifted_test.clip(lower=0)  # 🔥 음수 방지
                    self.df_test[f'{col}_sqrt'] = np.sqrt(shifted_test)
                
                print(f"✅ {col}: Sqrt 변환 적용 (Shift: {-min_value+1:.2f}, Train & Test)")
        
        return self.df_train, self.df_test
    
    @timer
    def encode_categorical_cv_safe(self, n_splits=None):
        """Target Encoding (CV-Safe, Train → Test)"""
        if n_splits is None:
            n_splits = CFG.CV_FOLDS
        
        print("\n" + "="*50)
        print("🔧 범주형 인코딩 시작 (CV-Safe)")
        print("="*50)
        
        if CFG.ENCODING_METHOD != 'target':
            return self._encode_categorical_simple()
    
        print(f"🎯 Target Encoding with {n_splits}-Fold Cross-Validation")
        
        # 🆕 ID/GROUP 컬럼 제외 (Target Encoding 대상에서 제외)
        categorical_cols_to_encode = [
            col for col in self.eda.categorical_cols 
            if col not in [CFG.ID_COL, CFG.GROUP_COL]
        ]
        
        print(f"📋 인코딩 대상: {len(categorical_cols_to_encode)}개 컬럼")
        if CFG.ID_COL:
            print(f"🚫 제외: ID 컬럼 '{CFG.ID_COL}'")
        if CFG.GROUP_COL:
            print(f"🚫 제외: GROUP 컬럼 '{CFG.GROUP_COL}'")
    
        # ===== Train: Fold별 Target Encoding =====
        if CFG.USE_GROUP_KFOLD and CFG.GROUP_COL in self.df_train.columns:
            print(f"🔄 GroupKFold 사용 (그룹: {CFG.GROUP_COL})")
            kfold = GroupKFold(n_splits=n_splits)
            splits = kfold.split(self.df_train, self.df_train[self.target_col], 
                                groups=self.df_train[CFG.GROUP_COL])
        elif self.eda.is_classification:
            kfold = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=CFG.RANDOM_STATE)
            splits = kfold.split(self.df_train, self.df_train[self.target_col])
        else:
            kfold = KFold(n_splits=n_splits, shuffle=True, random_state=CFG.RANDOM_STATE)
            splits = kfold.split(self.df_train)
        
        # 🔥 수정: categorical_cols_to_encode 사용
        for col in categorical_cols_to_encode:
            global_mean = self.df_train[self.target_col].mean()
            encoded_values = np.full(len(self.df_train), global_mean)
        
            # Fold별 인코딩
            for fold_idx, (train_idx, val_idx) in enumerate(splits):
                train_fold = self.df_train.iloc[train_idx]
                
                category_means = train_fold.groupby(col)[self.target_col].mean()
                category_counts = train_fold[col].value_counts()
                
                C = CFG.TARGET_ENCODING_SMOOTHING
                smoothed_means = ((category_means * category_counts) + (global_mean * C)) / (category_counts + C)
                
                val_categories = self.df_train.iloc[val_idx][col]
                encoded_values[val_idx] = val_categories.map(smoothed_means).fillna(global_mean).values
            
            self.df_train[f'{col}_target_enc'] = encoded_values
            
            # Test용 Mapping 저장
            full_category_means = self.df_train.groupby(col)[self.target_col].mean()
            full_category_counts = self.df_train[col].value_counts()
            full_smoothed_means = ((full_category_means * full_category_counts) + (global_mean * C)) / (full_category_counts + C)
            
            self.stats['target_encoding_maps'][col] = (full_smoothed_means, global_mean)
            
            self.df_train.drop(columns=[col], inplace=True)
            print(f"✅ [Train] {col}: Target Encoding 완료")
    
        # ===== Test 처리 =====
        if self.df_test is not None:
            print("\n📦 Test 데이터 Target Encoding (Train Mapping 적용)")
            for col in categorical_cols_to_encode:
                if col in self.df_test.columns:
                    mapping, global_mean = self.stats['target_encoding_maps'][col]
                    self.df_test[f'{col}_target_enc'] = self.df_test[col].map(mapping).fillna(global_mean)
                    self.df_test.drop(columns=[col], inplace=True)
                    print(f"✅ [Test] {col}: Target Encoding 적용")
        
        # 🆕 ID/GROUP 컬럼 제거 (모델링에 사용 안 함)
        cols_to_drop = []
        if CFG.ID_COL and CFG.ID_COL in self.df_train.columns:
            cols_to_drop.append(CFG.ID_COL)
        if CFG.GROUP_COL and CFG.GROUP_COL in self.df_train.columns:
            cols_to_drop.append(CFG.GROUP_COL)
        
        if cols_to_drop:
            self.df_train.drop(columns=cols_to_drop, inplace=True, errors='ignore')
            if self.df_test is not None:
                self.df_test.drop(columns=cols_to_drop, inplace=True, errors='ignore')
            print(f"\n🗑️ ID/GROUP 컬럼 제거 (전처리용): {cols_to_drop}")
        
        return self.df_train, self.df_test
    
    def _encode_categorical_simple(self):
        """Target Encoding 외 인코딩 (Train/Test 동일)"""
        for col in self.eda.categorical_cols:
            if CFG.ENCODING_METHOD == 'onehot':
                # One-Hot Encoding
                if self.df_train[col].nunique() <= CFG.ONEHOT_MAX_CARDINALITY:
                    # Train
                    train_dummies = pd.get_dummies(self.df_train[col], prefix=col, drop_first=True)
                    self.df_train = pd.concat([self.df_train, train_dummies], axis=1)
                    self.df_train.drop(columns=[col], inplace=True)
                    
                    # Test
                    if self.df_test is not None and col in self.df_test.columns:
                        test_dummies = pd.get_dummies(self.df_test[col], prefix=col, drop_first=True)
                        # Train에만 있는 컬럼은 Test에 0으로 추가
                        for train_col in train_dummies.columns:
                            if train_col not in test_dummies.columns:
                                test_dummies[train_col] = 0
                        # Test에만 있는 컬럼은 제거
                        test_dummies = test_dummies[train_dummies.columns]
                        
                        self.df_test = pd.concat([self.df_test, test_dummies], axis=1)
                        self.df_test.drop(columns=[col], inplace=True)
                    
                    print(f"✅ {col}: One-Hot Encoding 적용 (Train & Test)")
            
            elif CFG.ENCODING_METHOD == 'label':
                # Label Encoding
                if self.df_test is not None and col in self.df_test.columns:
                    all_categories = pd.concat([self.df_train[col], self.df_test[col]])
                else:
                    all_categories = self.df_train[col]
                
                category_mapping = {cat: idx for idx, cat in enumerate(all_categories.unique())}
                
                self.df_train[col] = self.df_train[col].map(category_mapping)
                if self.df_test is not None and col in self.df_test.columns:
                    self.df_test[col] = self.df_test[col].map(category_mapping).fillna(-1)
                
                print(f"✅ {col}: Label Encoding 적용 (Train & Test)")
            
            elif CFG.ENCODING_METHOD == 'frequency':
                # Frequency Encoding
                freq = self.df_train[col].value_counts(normalize=True)
                self.df_train[f'{col}_freq'] = self.df_train[col].map(freq)
                self.df_train.drop(columns=[col], inplace=True)
                
                if self.df_test is not None and col in self.df_test.columns:
                    # Test의 새로운 카테고리는 0으로
                    self.df_test[f'{col}_freq'] = self.df_test[col].map(freq).fillna(0)
                    self.df_test.drop(columns=[col], inplace=True)
                
                print(f"✅ {col}: Frequency Encoding 적용 (Train & Test)")
        
        return self.df_train, self.df_test
    
    @timer
    def scale_features(self):
        """스케일링 (Train Scaler → Test 적용)"""
        print("\n" + "="*50)
        print("🔧 Feature 스케일링 시작")
        print("="*50)
        
        if CFG.SCALING_METHOD is None:
            print("⚠️ 스케일링 비활성화")
            return self.df_train, self.df_test
        
        from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
        
        numerical_cols = self.df_train.select_dtypes(include=['number']).columns.tolist()
        numerical_cols = [c for c in numerical_cols if c not in [self.target_col, CFG.GROUP_COL]]
        
        if CFG.SCALING_METHOD == 'standard':
            scaler = StandardScaler()
        elif CFG.SCALING_METHOD == 'minmax':
            scaler = MinMaxScaler()
        elif CFG.SCALING_METHOD == 'robust':
            scaler = RobustScaler()
        
        # Train Fit & Transform
        self.df_train[numerical_cols] = scaler.fit_transform(self.df_train[numerical_cols])
        self.stats['scaler'] = scaler
        print(f"✅ [Train] {len(numerical_cols)}개 변수 스케일링 완료")
        
        # Test Transform (Train Scaler 사용)
        if self.df_test is not None:
            test_numerical_cols = [c for c in numerical_cols if c in self.df_test.columns]
            self.df_test[test_numerical_cols] = scaler.transform(self.df_test[test_numerical_cols])
            print(f"✅ [Test] {len(test_numerical_cols)}개 변수 스케일링 완료 (Train Scaler 적용)")
        
        return self.df_train, self.df_test
    
    @timer
    def select_features(self):
        """Feature Selection (Train → Test 동일 컬럼 제거)"""
        print("\n" + "="*50)
        print("🔧 Feature Selection 시작")
        print("="*50)
        
        if not CFG.FEATURE_SELECTION:
            print("⚠️ Feature Selection 비활성화")
            return self.df_train, self.df_test
        
        weak_features = []
        
        if hasattr(self.eda, 'feature_importance') and not self.eda.feature_importance.empty:
            for _, row in self.eda.feature_importance.iterrows():
                p_value_str = row['P-value']
                # Scientific notation 처리
                if isinstance(p_value_str, str):
                    p_value = float(p_value_str)
                else:
                    p_value = p_value_str
                    
                if p_value > CFG.FEATURE_SELECTION_THRESHOLD:
                    weak_features.append(row['Feature'])
            
            if weak_features:
                self.df_train.drop(columns=weak_features, inplace=True, errors='ignore')
                if self.df_test is not None:
                    self.df_test.drop(columns=weak_features, inplace=True, errors='ignore')
                print(f"✅ 예측력 낮은 Feature {len(weak_features)}개 제거 (Train & Test)")
            else:
                print("✅ 모든 Feature가 유의미합니다 (P-value < 0.05)")
        
        return self.df_train, self.df_test
    
    @timer
    def run_full_pipeline(self, n_splits=None):
        """전체 파이프라인 실행 (Train + Test)"""
        if n_splits is None:
            n_splits = CFG.CV_FOLDS
        
        print("\n" + "🚀"*25)
        print("자동 전처리 파이프라인 시작 (Train & Test)")
        print("🚀"*25)
        
        self.handle_missing_values()
        self.handle_outliers()
        self.create_group_features()
        self.apply_transformations()
        self.encode_categorical_cv_safe(n_splits=n_splits)
        self.scale_features()
        self.select_features()
        
        # 🆕 ID 컬럼 복원 (조건부: 그룹화 안 했을 때만)
        if not CFG.AGGREGATE_BY_GROUP:
            if self.train_id is not None:
                self.df_train.insert(0, CFG.ID_COL, self.train_id)
                print(f"\n✅ Train ID 컬럼 '{CFG.ID_COL}' 복원 완료 (첫 번째 컬럼)")
            
            if self.df_test is not None and self.test_id is not None:
                self.df_test.insert(0, CFG.ID_COL, self.test_id)
                print(f"✅ Test ID 컬럼 '{CFG.ID_COL}' 복원 완료 (첫 번째 컬럼)")
        else:
            print(f"\n🚫 그룹화 활성화: ID 컬럼 복원 건너뜀")
            print(f"   → GROUP 컬럼 '{CFG.GROUP_COL}'이 이미 집계 단위로 사용됨")
            print(f"   → 최종 데이터는 {CFG.GROUP_COL} 단위로 저장됨")
        
        # ===== 저장 =====
        train_path = os.path.join(CFG.PREPROCESS_DIR, "preprocessed_train.csv")
        self.df_train.to_csv(train_path, index=False, encoding='utf-8-sig')
        print(f"\n✅ Train 저장: {train_path}")
        print(f"📊 Train shape: {self.df_train.shape}")
        print(f"📋 Train columns (first 5): {list(self.df_train.columns[:5])}")
        
        if self.df_test is not None:
            test_path = os.path.join(CFG.PREPROCESS_DIR, "preprocessed_test.csv")
            self.df_test.to_csv(test_path, index=False, encoding='utf-8-sig')
            print(f"✅ Test 저장: {test_path}")
            print(f"📊 Test shape: {self.df_test.shape}")
            print(f"📋 Test columns (first 5): {list(self.df_test.columns[:5])}")
        
        return self.df_train, self.df_test


# ====================================================
# 8️⃣ 실행부 (Main)
# ====================================================
if __name__ == "__main__":
    # 1. 데이터 로딩 (Train + Test + 그룹화 옵션)
    df_train, df_test = get_full_dataset()
    
    if df_train is not None:
        # 2. EDA (Train만)
        eda = EDAPipeline(df_train, target_col=CFG.TARGET_COL)
        eda.show_basic_info()
        eda.analyze_missing_values()
        eda.detect_outliers()
        eda.analyze_numerical_distributions()
        eda.analyze_correlation()
        eda.analyze_cardinality()
        eda.analyze_feature_importance()
        eda.plot_target_distributions()
        
        print("\n" + "="*50)
        print("✅ EDA 파이프라인 완료!")
        print("="*50)
        
        # 3. 전처리 (Train + Test 함께)
        if CFG.AUTO_PREPROCESSING:
            preprocessor = AutoPreprocessor(df_train, eda, df_test=df_test)
            df_train_processed, df_test_processed = preprocessor.run_full_pipeline()
            
            print("\n" + "="*50)
            print("✅ 전처리 파이프라인 완료!")
            print("="*50)
            print("\n📌 다음 단계: 모델링")
            print("전처리된 데이터 경로:")
            print(f"  - Train: {os.path.join(CFG.PREPROCESS_DIR, 'preprocessed_train.csv')}")
            if df_test_processed is not None:
                print(f"  - Test: {os.path.join(CFG.PREPROCESS_DIR, 'preprocessed_test.csv')}")

📂 데이터 로딩 시작: C:\SEMIN\Project\ML_FINAL\1.BASE
✅ Train-Target 병합 완료 (Key: custid)

👥 custid로 데이터 그룹화 시작
  → 원본: 1,036,653 행 (거래 단위)
  → 기본 집계 중... (9개 컬럼)
✅ Train-Target 병합 완료 (Key: custid)

👥 custid로 데이터 그룹화 시작
  → 원본: 1,036,653 행 (거래 단위)
  → 기본 집계 중... (9개 컬럼)
  → One-Hot 집계 중... (8개 컬럼)
    ✅ sales_date: onehot_sum 생성 (10개 컬럼)
  → One-Hot 집계 중... (8개 컬럼)
    ✅ sales_date: onehot_sum 생성 (10개 컬럼)
    ✅ str_nm: onehot_sum 생성 (4개 컬럼)
    ✅ brd_nm: onehot_sum 생성 (10개 컬럼)
    ✅ str_nm: onehot_sum 생성 (4개 컬럼)
    ✅ brd_nm: onehot_sum 생성 (10개 컬럼)
    ✅ corner_nm: onehot_sum 생성 (10개 컬럼)
    ✅ pc_nm: onehot_sum 생성 (10개 컬럼)
    ✅ corner_nm: onehot_sum 생성 (10개 컬럼)
    ✅ pc_nm: onehot_sum 생성 (10개 컬럼)
    ✅ part_nm: onehot_sum 생성 (10개 컬럼)
    ✅ team_nm: onehot_sum 생성 (5개 컬럼)
    ✅ part_nm: onehot_sum 생성 (10개 컬럼)
    ✅ team_nm: onehot_sum 생성 (5개 컬럼)
    ✅ buyer_nm: onehot_sum 생성 (10개 컬럼)
  → 결측치 처리 시작...
  ✅ 결측치 201,390개 → 0으로 채움
✅ 타겟 컬럼 'gender' 정상 유지
  → 결과: 30,000 행 (고객 단위)
  → 컬럼 수: 18 → 119

📊 

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                   Categories                                    │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓                        │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃ ┃ Categorical Variables         ┃                        │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩                        │
│ │ Number of rows    │ 30000  │ │ float64     │ 76    │ │ import_flg_min                │                        │
│ │ Number of columns │ 118    │ │ int64       │ 35    │ │ import_flg_max                │                        │
│ └───────────────────┴────────┘ │ category    │ 7     │ │ inst_mon_min                  │                        │
│                                └─────────────┴───────┘ │ inst_fee_min                  │                        │
│                                                        │ inst_fee_max                  │                        │
│                                                        │ team_nm_count_상품개발영업2과 │                        │
│                                                        │ team_nm_count_인터넷백화점    │                        │
│                                                        └───────────────────────────────┘                        │
│                                                     number                                                      │
│ ┏━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┓  │
│ ┃ column   ┃ NA ┃ NA % ┃ mean     ┃ sd       ┃ p0       ┃ p25      ┃ p50      ┃ p75      ┃ p100     ┃ hist   ┃  │
│ ┡━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━┩  │
│ │ sales_ti │  0 │    0 │     1560 │    143.7 │      978 │     1466 │     1565 │     1658 │     2023 │  ▁▅▇▃  │  │
│ │ me_mean  │    │      │          │          │          │          │          │          │          │        │  │
│ │ sales_ti │  0 │    0 │    194.2 │     71.9 │        0 │    158.8 │    201.6 │    239.2 │      688 │  ▂▇▅   │  │
│ │ me_std   │    │      │          │          │          │          │          │          │          │        │  │
│ │ sales_ti │  0 │    0 │    53390 │    61560 │     1023 │    14920 │    33240 │    67700 │   863300 │   ▇▁   │  │
│ │ me_sum   │    │      │          │          │          │          │          │          │          │        │  │
│ │ sales_ti │  0 │    0 │     1218 │    188.8 │       23 │     1100 │     1150 │     1313 │     2023 │    ▇▁  │  │
│ │ me_min   │    │      │          │          │          │          │          │          │          │        │  │
│ │ sales_ti │  0 │    0 │     1857 │      145 │     1023 │     1820 │     1910 │     1940 │     2347 │   ▁▅▇  │  │
│ │ me_max   │    │      │          │          │          │          │          │          │          │        │  │
│ │ sales_ti │  0 │    0 │    34.56 │    40.06 │        1 │       10 │       21 │       44 │      589 │   ▇    │  │
│ │ me_count │    │      │          │          │          │          │          │          │          │        │  │
│ │ goodcd_m │  0 │    0 │ 39350000 │ 45530000 │ 21160000 │ 37090000 │ 40210000 │ 42190000 │ 62840000 │   ▂▇▃  │  │
│ │ ean      │    │      │    00000 │     0000 │    00000 │    00000 │    00000 │    00000 │    00000 │        │  │
│ │ goodcd_s │  0 │    0 │ 79440000 │ 35000000 │        0 │ 64750000 │ 83230000 │ 99360000 │ 30180000 │  ▂▇▃   │  │
│ │ td       │    │      │     0000 │     0000 │          │     0000 │     0000 │     0000 │    00000 │        │  │
│ │ goodcd_s │  0 │    0 │ 13520000 │ 15280000 │ 21160000 │ 37850000 │ 85390000 │ 17430000 │ 20080000 │   ▇▁   │  │
│ │ um       │    │      │  0000000 │  0000000 │    00000 │   000000 

⏱️ show_basic_info 실행 시간: 0.44초

3. 결측치(Missing Values) 분석
✅ 결측치가 없습니다!
⏱️ analyze_missing_values 실행 시간: 0.01초

4. 이상치(Outliers) 탐지 (Method: IQR)
                             Feature  Outlier_Count  Outlier_Percent           Min          Max   Lower_Bound  Upper_Bound
                        inst_mon_max           9012            30.04  1.000000e+00 1.200000e+01  3.000000e+00 3.000000e+00
                    pc_nm_count_진캐주얼           7350            24.50  0.000000e+00 6.900000e+01  0.000000e+00 0.000000e+00
                        inst_mon_std           6941            23.14  0.000000e+00 7.778175e+00  3.651085e-01 1.467483e+00
                  part_nm_count_영플라자           6918            23.06  0.000000e+00 1.450000e+02  0.000000e+00 0.000000e+00
                     brd_nm_count_랑콤           6805            22.68  0.000000e+00 3.400000e+01  0.000000e+00 0.000000e+00
                 corner_nm_count_캐릭터           6760            22.53  0.000000e+00 1.530000e+02  0.000000e+00 0.0000